# Pipeline 3 — Temporal Particle Tracking End-to-End Test

This notebook processes a multi-frame sequence through the full Pipeline 3:
- Frame 0: Pipeline 1 cold-start (BestFirstSearcher)
- Frame 1+: IMU-predicted particle-guided two-pass search + meta-tile + semantic confirmation

Outputs: JSONL logs, meta-tiles, trajectory CSV, summary metrics.

In [ ]:
# Cell 1 — Setup
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from config import config
from src.tile_utils import TileLoader, haversine_distance
from src.image_utils import load_image
from src.geometric_matcher import initialize_matcher
from src.semantic_model import load_semantic_model
from src.temporal_searcher import TemporalSearcher
from src.ekf_ins import preprocess_imu_csv

config.ensure_output_dirs()
print('Setup complete')

In [ ]:
# Cell 2 — Load data + warmup ONLINE EKF to START_ROW
from src.ekf_ins import preprocess_imu_csv, ErrorStateEKF, step_ekf, barometric_altitude
from src.tile_utils import tile_to_latlon

# Run batch EKF for reference / GT comparison (unchanged)
imu_log = preprocess_imu_csv(config.IMU_CSV_PATH)
print(f'IMU log: {len(imu_log)} rows')

# Build frame list: match CSV timestamps to frame files
frame_dir = config.QUERY_FRAMES_DIR
frame_files = sorted(frame_dir.glob('frame_*.jpg'))
print(f'Found {len(frame_files)} frames')

frame_map = {}
for fp in frame_files:
    ts_str = fp.stem.replace('frame_', '')
    try:
        ts = float(ts_str)
        frame_map[round(ts, 3)] = fp
    except ValueError:
        continue

# Reference map bounds (for diagnostics)
south_lat, west_lon = tile_to_latlon(config.TILE_X_MIN, config.TILE_Y_MIN, config.TMS_ZOOM_LEVEL)
north_lat, _ = tile_to_latlon(config.TILE_X_MIN, config.TILE_Y_MAX + 1, config.TMS_ZOOM_LEVEL)
_, east_lon = tile_to_latlon(config.TILE_X_MAX + 1, config.TILE_Y_MIN, config.TMS_ZOOM_LEVEL)
map_bounds = {
    'lat_min': south_lat, 'lat_max': north_lat,
    'lon_min': west_lon, 'lon_max': east_lon,
}
print(f'Map bounds: lat [{map_bounds["lat_min"]:.4f}, {map_bounds["lat_max"]:.4f}], '
      f'lon [{map_bounds["lon_min"]:.4f}, {map_bounds["lon_max"]:.4f}]')

# Process ALL frames from the beginning
START_ROW = 0
NUM_FRAMES = len(imu_log)
print(f'START_ROW = {START_ROW}  |  NUM_FRAMES = {NUM_FRAMES}  (full flight)')

# ── Warmup: create a LIVE EKF at row 0 (cold start) ──
raw_df = pd.read_csv(config.IMU_CSV_PATH)
lat0 = raw_df['latitude'].iloc[0]
lon0 = raw_df['longitude'].iloc[0]
alt0 = barometric_altitude(raw_df['barometer_pressure'].iloc[0])
heading0 = np.degrees(raw_df['heading_magnetic'].iloc[0])
airspeed0 = raw_df['airspeed_true'].iloc[0] if 'airspeed_true' in raw_df.columns else None

live_ekf = ErrorStateEKF(lat0, lon0, alt0, heading0, airspeed0)

prev_ts = None
for i in range(START_ROW + 1):  # include START_ROW itself
    row_dict = raw_df.iloc[i].to_dict()
    step_ekf(live_ekf, row_dict, prev_ts)
    prev_ts = row_dict['timestamp']

s0 = live_ekf.get_state()
print(f'Live EKF initialized at row {START_ROW}:')
print(f'  Position: ({s0["latitude"]:.6f}, {s0["longitude"]:.6f})')
print(f'  Heading:  {s0["yaw"]:.1f}°')
print(f'  P_pos diag: [{live_ekf.P[8,8]:.0f}, {live_ekf.P[9,9]:.0f}] m²')

# Align ALL frames — no cap
aligned = []
for idx in range(START_ROW, len(imu_log)):
    row = imu_log.iloc[idx]
    ts_rounded = round(row['timestamp'], 3)
    if ts_rounded in frame_map:
        aligned.append((idx, row['timestamp'], frame_map[ts_rounded]))

print(f'Aligned {len(aligned)} frame-IMU pairs (full flight)')
if aligned:
    r0 = imu_log.iloc[aligned[0][0]]
    print(f'First frame GPS: ({r0["gps_lat"]:.6f}, {r0["gps_lon"]:.6f})')
    print(f'First frame EKF: ({r0["latitude_est"]:.6f}, {r0["longitude_est"]:.6f})')

In [ ]:

# Cell 3 — Initialize Temporal Searcher (re-imports after module reload)
from config import config
from src.tile_utils import TileLoader
from src.image_utils import load_image
from src.geometric_matcher import initialize_matcher
from src.semantic_model import load_semantic_model
from src.temporal_searcher import TemporalSearcher

semantic_model = load_semantic_model(config.SEMANTIC_MODEL_PATH, config.DEVICE)
matcher = initialize_matcher(config.DEVICE, config.MAX_NUM_KEYPOINTS)
tile_loader = TileLoader(
    config.REFERENCE_TILES_DIR,
    config.REFERENCE_PRED_DIR,
    zoom=config.TMS_ZOOM_LEVEL,
    x_range=(config.TILE_X_MIN, config.TILE_X_MAX),
    y_range=(config.TILE_Y_MIN, config.TILE_Y_MAX),
)
print(f'Tiles available: {len(tile_loader.list_tiles())}')

# Load precomputed SuperPoint reference features (if available)
feature_store = None
if hasattr(config, 'REFERENCE_FEATURES_PATH') and config.REFERENCE_FEATURES_PATH.exists():
    sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
    from Dataset_Preprocessing.feature_store import FeatureStoreLoader
    feature_store = FeatureStoreLoader(config.REFERENCE_FEATURES_PATH, device=config.DEVICE)
    feature_store.open()   # ← MUST call open() to load HDF5 and build tile index
    print(f'Feature store loaded: {feature_store.num_tiles} precomputed tiles')
    print(f'  HDF5 path: {config.REFERENCE_FEATURES_PATH}')
    meta = feature_store.metadata or {}
    if meta:
        print(f'  extractor: {meta.get("extractor_name","?")}  '
              f'max_kp: {meta.get("max_keypoints","?")}  '
              f'dtype: {meta.get("descriptor_dtype","?")}')
else:
    print('No precomputed feature store found — will extract features at runtime')
    print(f'  Checked path: {getattr(config, "REFERENCE_FEATURES_PATH", "N/A")}')

searcher = TemporalSearcher(semantic_model, matcher, tile_loader, config,
                            feature_store=feature_store)
print('TemporalSearcher initialized')

# Verify new interfaces
assert hasattr(matcher, 'extract_features'), "matcher.extract_features missing!"
assert hasattr(matcher, 'match_precomputed'), "matcher.match_precomputed missing!"
print(f'  matcher has extract_features: OK')
print(f'  matcher has match_precomputed: OK')


In [ ]:
# Cell 4 — CLOSED-LOOP: EKF predict → visual match → EKF update
from src.tile_utils import haversine_distance as _hav
from src.ekf_ins import step_ekf
import copy, time, math
from pathlib import Path

RUN_N = len(aligned)  # process ALL frames

# Adaptive measurement noise: trust high-quality measurements more
R_HIGH  = 30.0 ** 2   # CShape>0.5 AND inliers>100 → 900 m²
R_MED   = 60.0 ** 2   # CShape>0.3 AND inliers>20  → 3600 m²

# Camera look-ahead correction (homo position overshoots in heading direction)
LOOKAHEAD_M = 110.0   # empirically measured from offset analysis

# During heavy bank turns the visual measurement is from an oblique view;
# inflate R so a bad measurement doesn't corrupt the tight EKF covariance.
TURN_ROLL_THRESHOLD_RAD = 0.26  # ~15 degrees
TURN_R_MULTIPLIER = 3.0

# Deep-copy the warmed-up EKF so we can re-run without re-warming
ekf = copy.deepcopy(live_ekf)
searcher.frame_count = 0  # reset searcher state
searcher.particle_filter = None

results = []
ekf_errors_online = []
ekf_errors_batch  = []
homo_errors = []
trajectory_est = []   # (lat, lon) online EKF
trajectory_gt  = []   # (lat, lon) ground truth GPS
trajectory_batch = [] # (lat, lon) batch EKF dead reckoning
gate_count = 0
prev_csv_idx = START_ROW

# Track previous timestamp for step_ekf
prev_ts_ekf = raw_df.iloc[START_ROW]['timestamp']

# Header
print(f'{"F":>3s} | {"image":<26s} | {"final":>6s} {"batch":>6s} {"homo":>6s} | '
      f'{"CS":>5s} {"inl":>4s} {"SC":>5s} {"gate":>4s} {"R":>5s} | {"method":<20s} | {"t":>4s} | note')
print('-' * 140)

t0_total = time.perf_counter()
for i, (csv_idx, ts, frame_path) in enumerate(aligned[:RUN_N]):
    t0_frame = time.perf_counter()

    # ── Step A: Step online EKF on this row ──
    row_dict = raw_df.iloc[csv_idx].to_dict()
    step_ekf(ekf, row_dict, prev_ts_ekf)
    prev_ts_ekf = row_dict['timestamp']
    prev_csv_idx = csv_idx

    ekf_state = ekf.get_state()
    ekf_lat, ekf_lon = ekf_state['latitude'], ekf_state['longitude']
    ekf_yaw = ekf_state['yaw']

    # ── Step B: Visual matching ──
    query_frame = load_image(frame_path)
    vel = np.sqrt(ekf_state['vel_n']**2 + ekf_state['vel_e']**2)
    bank_rad = abs(row_dict.get('bank', 0.0))

    imu_data = {
        'lat': ekf_lat, 'lon': ekf_lon,
        'heading': ekf_yaw,
        'pos_sigma': np.sqrt(max(ekf.P[8, 8], ekf.P[9, 9])),
        'heading_sigma': 15.0,
        'velocity_mps': vel,
        'gyro_z_dps': row_dict.get('gyro_z', 0.0) * (180.0 / np.pi),
        'pitch': row_dict.get('pitch', 0.0),
        'roll': row_dict.get('bank', 0.0),
    }

    result = searcher.process_frame(query_frame, imu_data, timestamp=ts)
    results.append(result)

    # ── Step C: Quality-adaptive EKF position update ──
    gate_pass = result.get('gate_pass', False)
    homo_pos = result.get('homo_position')
    vq = result.get('visual_quality', {})
    cs = vq.get('CShape', 0)
    ni = vq.get('inliers', 0)
    sem_conf = result.get('semantic_confidence') or 0.5
    meta_verified = result.get('meta_tile_verified', False)

    note = ''
    r_used = None

    # Apply camera look-ahead correction: shift backward along heading
    if homo_pos is not None and LOOKAHEAD_M > 0:
        h_rad = math.radians(ekf_yaw)
        corr_north = -LOOKAHEAD_M * math.cos(h_rad)
        corr_east  = -LOOKAHEAD_M * math.sin(h_rad)
        homo_pos = (
            homo_pos[0] + corr_north / 111320.0,
            homo_pos[1] + corr_east / (111320.0 * math.cos(math.radians(homo_pos[0]))),
        )

    if gate_pass and homo_pos is not None:
        # Adaptive R: high quality → lower noise → more trust
        if cs > 0.5 and ni > 100:
            r_used = R_HIGH
        else:
            r_used = R_MED

        # Heavy bank turn: oblique view → inflate R to protect EKF covariance
        if bank_rad > TURN_ROLL_THRESHOLD_RAD:
            r_used *= TURN_R_MULTIPLIER
            note += f'turn({math.degrees(bank_rad):.0f}°) '

        # Meta-tile verification: if the meta-tile didn't pass the match
        # threshold (≥25 keypoints), the homography may be unreliable —
        # inflate R by 2× so the EKF trusts it less.
        if not meta_verified:
            r_used *= 2.0
            note += 'unverified '

        # Semantic confidence soft-modulation: high sem_conf → more trust, low → less
        r_used *= max(0.5, 2.0 - 1.5 * sem_conf)

        ekf.update_position(homo_pos[0], homo_pos[1], R_pos_m2=r_used)
        gate_count += 1
    else:
        # Explain WHY the gate failed
        reasons = []
        if homo_pos is None:
            reasons.append('no_homo')
        if cs <= 0.3:
            reasons.append(f'CS={cs:.2f}<0.3')
        if ni <= 20:
            reasons.append(f'inl={ni}<20')
        note = 'SKIP: ' + ', '.join(reasons) if reasons else 'SKIP'

    # ── Step D: Final corrected position ──
    corrected = ekf.get_state()
    final_lat, final_lon = corrected['latitude'], corrected['longitude']

    # Ground truth
    gt_row = imu_log.iloc[csv_idx]
    gt_lat, gt_lon = gt_row['gps_lat'], gt_row['gps_lon']

    final_err = _hav(final_lat, final_lon, gt_lat, gt_lon)
    batch_err = _hav(gt_row['latitude_est'], gt_row['longitude_est'], gt_lat, gt_lon)
    ekf_errors_online.append(final_err)
    ekf_errors_batch.append(batch_err)
    trajectory_est.append((final_lat, final_lon))
    trajectory_gt.append((gt_lat, gt_lon))
    trajectory_batch.append((gt_row['latitude_est'], gt_row['longitude_est']))

    homo_err = _hav(homo_pos[0], homo_pos[1], gt_lat, gt_lon) if homo_pos else None
    if homo_err is not None:
        homo_errors.append(homo_err)

    # ── Print EVERY frame ──
    fname = Path(frame_path).stem
    g_str = 'PASS' if gate_pass else 'fail'
    h_str = f'{homo_err:5.0f}m' if homo_err is not None else '  N/A'
    r_str = f'{np.sqrt(r_used):4.0f}' if r_used else '   -'
    sc_str = f'{sem_conf:.2f}'
    t_frame = time.perf_counter() - t0_frame
    beat = '✓' if final_err < batch_err else ' '

    print(f'F{i:3d} | {fname:<26s} | {final_err:5.0f}m {batch_err:5.0f}m {h_str} | '
          f'{cs:.3f} {ni:4d} {sc_str:>5s} {g_str:>4s} {r_str} | {result["method"]:<20s} | '
          f'{t_frame:4.1f}s | {note}{beat}')

elapsed = time.perf_counter() - t0_total

# ═══════════════════════════════════════════════════════════════ SUMMARY ═══
print(f'\n{"="*140}')
print(f'Processed {len(results)} frames in {elapsed:.1f}s ({elapsed/len(results):.1f}s/frame)')
print(f'\n  Online EKF (visual-corrected):  mean={np.mean(ekf_errors_online):.1f}m  '
      f'median={np.median(ekf_errors_online):.1f}m  min={np.min(ekf_errors_online):.1f}m  '
      f'max={np.max(ekf_errors_online):.1f}m')
print(f'  Batch EKF  (dead-reckoned):     mean={np.mean(ekf_errors_batch):.1f}m')
impr = np.mean(ekf_errors_batch) - np.mean(ekf_errors_online)
print(f'  Improvement: {impr:.1f}m ({impr/np.mean(ekf_errors_batch)*100:.1f}%)')
print(f'\n  Quality gate: {gate_count}/{len(results)} passed ({gate_count/len(results)*100:.0f}%)')
if homo_errors:
    print(f'  Homo-only:    mean={np.mean(homo_errors):.1f}m  min={np.min(homo_errors):.1f}m  '
          f'max={np.max(homo_errors):.1f}m  (n={len(homo_errors)})')
better = sum(1 for e, b in zip(ekf_errors_online, ekf_errors_batch) if e < b)
print(f'  Beating batch: {better}/{len(results)} ({better/len(results)*100:.0f}%)')
print(f'  <50m: {sum(1 for e in ekf_errors_online if e<50)}/{len(results)}  '
      f'<100m: {sum(1 for e in ekf_errors_online if e<100)}/{len(results)}  '
      f'<150m: {sum(1 for e in ekf_errors_online if e<150)}/{len(results)}')

print(f'  P_pos final: [{ekf.P[8,8]:.0f}, {ekf.P[9,9]:.0f}] m²')
from collections import Counter
print(f'  Methods: {dict(Counter(r["method"] for r in results))}')


In [ ]:
# Quick recap of closed-loop results
print(f'Online EKF mean:  {np.mean(ekf_errors_online):.1f}m  median: {np.median(ekf_errors_online):.1f}m')
print(f'Batch EKF mean:   {np.mean(ekf_errors_batch):.1f}m')
print(f'Improvement:      {np.mean(ekf_errors_batch) - np.mean(ekf_errors_online):.1f}m')
print(f'Gate passes:      {gate_count}/{len(results)}')
if homo_errors:
    print(f'Homo-only mean:   {np.mean(homo_errors):.1f}m (n={len(homo_errors)})')
print(f'Frames < 50m:     {sum(1 for e in ekf_errors_online if e < 50)}')
print(f'Frames < 100m:    {sum(1 for e in ekf_errors_online if e < 100)}')
print(f'Frames < 150m:    {sum(1 for e in ekf_errors_online if e < 150)}')
print(f'P_pos final:      [{ekf.P[8,8]:.0f}, {ekf.P[9,9]:.0f}]')
# Show first/last few errors
print(f'\nFirst 10 online errors: {[f"{e:.0f}" for e in ekf_errors_online[:10]]}')
print(f'Last 10 online errors:  {[f"{e:.0f}" for e in ekf_errors_online[-10:]]}')
print(f'First 10 batch errors:  {[f"{e:.0f}" for e in ekf_errors_batch[:10]]}')
# Frames where online is worse than batch
worse = [(i, o, b) for i, (o, b) in enumerate(zip(ekf_errors_online, ekf_errors_batch)) if o > b]
print(f'\nFrames where online WORSE than batch: {len(worse)}/{len(results)}')
if worse[:5]:
    for fi, o, b in worse[:5]:
        print(f'  F{fi}: online={o:.0f}m batch={b:.0f}m diff={o-b:+.0f}m')

In [ ]:
# Cell 6 — Trajectory Visualization: Estimation vs Ground Truth
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

est_lats  = [p[0] for p in trajectory_est]
est_lons  = [p[1] for p in trajectory_est]
gt_lats   = [p[0] for p in trajectory_gt]
gt_lons   = [p[1] for p in trajectory_gt]
bat_lats  = [p[0] for p in trajectory_batch]
bat_lons  = [p[1] for p in trajectory_batch]

# ─── Figure 1: Map view ───
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

ax = axes[0]
ax.plot(gt_lons, gt_lats, 'g-', lw=2.5, label='Ground Truth GPS', zorder=3)
ax.plot(est_lons, est_lats, 'b-', lw=1.8, label='Online EKF (ours)', zorder=4)
ax.plot(bat_lons, bat_lats, 'r--', lw=1.2, alpha=0.6, label='Batch EKF (dead reckoning)', zorder=2)

# Start/end markers
ax.scatter(gt_lons[0], gt_lats[0], c='green', s=120, marker='^', zorder=5, edgecolors='k', label='Start')
ax.scatter(gt_lons[-1], gt_lats[-1], c='red', s=120, marker='v', zorder=5, edgecolors='k', label='End')

# Reference map bounds
ax.axhline(map_bounds['lat_min'], color='gray', ls=':', alpha=0.3)
ax.axhline(map_bounds['lat_max'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_min'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_max'], color='gray', ls=':', alpha=0.3)

ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('Trajectory: Estimation vs Ground Truth', fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# ─── Figure 2: Error over time ───
ax2 = axes[1]
frames = list(range(len(ekf_errors_online)))
ax2.plot(frames, ekf_errors_online, 'b-', lw=1.5, label='Online EKF error')
ax2.plot(frames, ekf_errors_batch, 'r--', lw=1, alpha=0.6, label='Batch EKF error')
ax2.axhline(50, color='gold', ls='--', lw=1.5, alpha=0.7, label='50m target')
ax2.axhline(100, color='orange', ls='--', lw=1, alpha=0.5, label='100m')
ax2.fill_between(frames, 0, 50, color='green', alpha=0.08)
ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('Error (m)', fontsize=12)
ax2.set_title('Localization Error Over Time', fontsize=14)
ax2.legend(fontsize=10, loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/trajectory_vs_ground_truth.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: outputs/trajectory_vs_ground_truth.png")

# ─── Figure 3: Zoomed estimation close-up ───
fig2, ax3 = plt.subplots(figsize=(12, 10))

# Draw lines connecting estimation to GT for each frame (error vectors)
for i in range(0, len(est_lats), max(1, len(est_lats)//30)):
    ax3.plot([est_lons[i], gt_lons[i]], [est_lats[i], gt_lats[i]],
             'k-', alpha=0.15, lw=0.8)

ax3.plot(gt_lons, gt_lats, 'g-o', ms=3, lw=2, label='Ground Truth', zorder=3)
ax3.plot(est_lons, est_lats, 'b-o', ms=3, lw=1.5, label='Online EKF', zorder=4)
ax3.scatter(gt_lons[0], gt_lats[0], c='green', s=150, marker='^', zorder=5, edgecolors='k')
ax3.scatter(gt_lons[-1], gt_lats[-1], c='red', s=150, marker='v', zorder=5, edgecolors='k')

ax3.set_xlabel('Longitude (°E)', fontsize=12)
ax3.set_ylabel('Latitude (°N)', fontsize=12)
ax3.set_title('Online EKF vs Ground Truth (zoom)', fontsize=14)
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')
ax3.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))
plt.setp(ax3.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('../outputs/trajectory_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: outputs/trajectory_zoomed.png")

In [ ]:
# Cell 6 — Diagnostic: visual measurement accuracy vs quality metrics
import matplotlib.pyplot as plt
from src.tile_utils import haversine_distance as _hav

# Collect per-frame visual quality data
diag = []
for i, (csv_idx, ts, frame_path) in enumerate(aligned[:len(results)]):
    r = results[i]
    gt_row = imu_log.iloc[csv_idx]
    gt_lat, gt_lon = gt_row['gps_lat'], gt_row['gps_lon']
    hp = r.get('homo_position')
    vq = r.get('visual_quality', {})
    he = _hav(hp[0], hp[1], gt_lat, gt_lon) if hp else None
    diag.append({
        'frame': i, 'homo_err': he,
        'CShape': vq.get('CShape', 0), 'inliers': vq.get('inliers', 0),
        'gate': r.get('gate_pass', False),
        'online_err': ekf_errors_online[i], 'batch_err': ekf_errors_batch[i],
    })

# Distribution of homo errors
hes = [d['homo_err'] for d in diag if d['homo_err'] is not None]
print(f"Homo errors: mean={np.mean(hes):.0f}m  median={np.median(hes):.0f}m  "
      f"min={np.min(hes):.0f}m  max={np.max(hes):.0f}m  std={np.std(hes):.0f}m")
print(f"  <50m: {sum(1 for e in hes if e<50)}  <100m: {sum(1 for e in hes if e<100)}  "
      f"<150m: {sum(1 for e in hes if e<150)}  <200m: {sum(1 for e in hes if e<200)}")

# Quality vs accuracy correlation
gated = [d for d in diag if d['gate']]
ungated = [d for d in diag if not d['gate']]
if gated:
    ghe = [d['homo_err'] for d in gated if d['homo_err']]
    print(f"\nGated (quality pass): mean homo err={np.mean(ghe):.0f}m (n={len(ghe)})")
if ungated:
    uhe = [d['homo_err'] for d in ungated if d['homo_err']]
    if uhe:
        print(f"Ungated (quality fail): mean homo err={np.mean(uhe):.0f}m (n={len(uhe)})")

# Best 10 vs worst 10 visual measurements
sorted_diag = sorted([d for d in diag if d['homo_err'] is not None], key=lambda d: d['homo_err'])
print(f"\nBest 10 visual measurements:")
for d in sorted_diag[:10]:
    print(f"  F{d['frame']:3d}: homo={d['homo_err']:5.0f}m  CS={d['CShape']:.3f}  inl={d['inliers']:4d}  "
          f"online={d['online_err']:5.0f}m  gate={'Y' if d['gate'] else 'N'}")
print(f"\nWorst 10 visual measurements:")
for d in sorted_diag[-10:]:
    print(f"  F{d['frame']:3d}: homo={d['homo_err']:5.0f}m  CS={d['CShape']:.3f}  inl={d['inliers']:4d}  "
          f"online={d['online_err']:5.0f}m  gate={'Y' if d['gate'] else 'N'}")

# Plot: errors over time + quality markers
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
ax.plot([d['frame'] for d in diag], [d['online_err'] for d in diag], 'b-o', ms=4, label='Online EKF')
ax.plot([d['frame'] for d in diag], [d['batch_err'] for d in diag], 'r--', alpha=0.5, label='Batch EKF')
he_frames = [d['frame'] for d in diag if d['homo_err']]
he_vals = [d['homo_err'] for d in diag if d['homo_err']]
ax.scatter(he_frames, he_vals, c='green', s=20, alpha=0.6, zorder=5, label='Homo raw')
ax.axhline(50, color='gold', ls='--', alpha=0.5, label='50m target')
ax.set_ylabel('Error (m)'); ax.legend(); ax.set_title('Localization Error Over Time')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot([d['frame'] for d in diag], [d['CShape'] for d in diag], 'g-o', ms=3, label='CShape')
ax.axhline(0.3, color='red', ls='--', alpha=0.5, label='Gate thresh')
ax.axhline(0.5, color='orange', ls='--', alpha=0.5, label='High-quality')
ax.set_ylabel('CShape'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot([d['frame'] for d in diag], [d['inliers'] for d in diag], 'm-o', ms=3, label='Inliers')
ax.axhline(20, color='red', ls='--', alpha=0.5, label='Gate thresh')
ax.axhline(100, color='orange', ls='--', alpha=0.5, label='High-quality')
ax.set_ylabel('Inliers'); ax.set_xlabel('Frame'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/phase_c_diagnostics.png', dpi=150)
plt.show()
print("Saved: outputs/phase_c_diagnostics.png")

In [ ]:

# Cell 8 — Camera look-ahead diagnosis: is the offset in the heading direction?
import math

# For each gated frame, compute: offset vector (homo → GT), heading vector, angle between them
offset_analysis = []
for i, (csv_idx, ts, frame_path) in enumerate(aligned[:len(results)]):
    r = results[i]
    hp = r.get('homo_position')
    if hp is None or not r.get('gate_pass', False):
        continue
    gt_row = imu_log.iloc[csv_idx]
    gt_lat, gt_lon = gt_row['gps_lat'], gt_row['gps_lon']

    # Offset vector: UNCORRECTED homo → GT (meters, NED convention)
    d_north = (gt_lat - hp[0]) * 111320
    d_east = (gt_lon - hp[1]) * 111320 * math.cos(math.radians(gt_lat))
    offset_dir = math.degrees(math.atan2(d_east, d_north)) % 360  # bearing of (homo → GT)
    offset_dist = math.sqrt(d_north**2 + d_east**2)

    # Batch EKF yaw (NED heading, degrees) — best available proxy for flight heading
    heading = float(gt_row['yaw_deg']) if 'yaw_deg' in imu_log.columns else 0.0

    # Angle between offset direction and heading (anti-aligned → 180° → camera looks forward)
    angle_diff = (offset_dir - heading + 180) % 360 - 180  # [-180, 180]

    offset_analysis.append({
        'frame': i, 'offset_dist': offset_dist,
        'offset_dir': offset_dir, 'heading': heading,
        'angle_diff': angle_diff, 'd_north': d_north, 'd_east': d_east
    })

# Summary
dists = [o['offset_dist'] for o in offset_analysis]
angles = [o['angle_diff'] for o in offset_analysis]
print(f"Gated frames analyzed: {len(offset_analysis)}")
print(f"Offset distance: mean={np.mean(dists):.0f}m  std={np.std(dists):.0f}m")
print(f"Offset angle (relative to heading): mean={np.mean(angles):.1f}°  std={np.std(angles):.1f}°")
print(f"  (0° = exactly forward, ±180° = behind drone)")
print()

# If mean angle is near 0°, the offset is in the heading direction → camera look-ahead confirmed
if abs(np.mean(angles)) < 30:
    print(">>> CONFIRMED: Offset is primarily in the heading direction!")
    print(">>> This is consistent with a fixed camera forward-tilt.")
    altitude = 365  # approximate altitude in meters
    tilt_est = math.degrees(math.atan(np.mean(dists) / altitude))
    print(f">>> Estimated camera tilt: {tilt_est:.1f}° (at {altitude}m altitude)")
    suggestion = np.mean(dists)
    print(f">>> Suggested LOOKAHEAD_M = {suggestion:.0f}m  (current: {LOOKAHEAD_M}m)")
else:
    print(f">>> Offset NOT aligned with heading. Mean angle = {np.mean(angles):.1f}°")
    print(">>> Look for other systematic bias sources.")

# Per-frame details for worst offsets
print(f"\nPer-frame offset details (worst 10):")
for o in sorted(offset_analysis, key=lambda x: x['offset_dist'], reverse=True)[:10]:
    print(f"  F{o['frame']:3d}: dist={o['offset_dist']:5.0f}m  "
          f"offset_bearing={o['offset_dir']:5.1f}°  heading={o['heading']:5.1f}°  "
          f"diff={o['angle_diff']:+6.1f}°")

# Plot offset vectors
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
for o in offset_analysis:
    hrad = math.radians(o['heading'])
    ax.arrow(0, 0, math.sin(hrad)*50, math.cos(hrad)*50,
             head_width=3, color='blue', alpha=0.15)
    ax.arrow(0, 0, o['d_east'], o['d_north'],
             head_width=3, color='red', alpha=0.3)
ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
ax.set_title('Red=Homo→GT offset, Blue=Heading direction')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_xlim(-200, 200); ax.set_ylim(-200, 200)
plt.tight_layout()
plt.show()


In [ ]:

# Cell 9 — Build per-frame DataFrame + Multi-Method Comprehensive Summary
import math
import numpy as np
import pandas as pd
from collections import Counter
from src.tile_utils import haversine_distance as _hav

def _correct_lookahead(pos, heading_deg, dist_m):
    """Shift a lat/lon position backward along heading by dist_m meters."""
    if pos is None:
        return None
    h = math.radians(heading_deg)
    cn = -dist_m * math.cos(h)
    ce = -dist_m * math.sin(h)
    lat_cos = math.cos(math.radians(pos[0]))
    return (pos[0] + cn / 111320.0,
            pos[1] + ce / (111320.0 * lat_cos))

# Build per-frame record
frame_data = []
for i, (csv_idx, ts, fp) in enumerate(aligned[:len(results)]):
    r = results[i]
    gt_row  = imu_log.iloc[csv_idx]
    raw_row = raw_df.iloc[csv_idx]

    gt_lat, gt_lon = float(gt_row['gps_lat']), float(gt_row['gps_lon'])
    heading_deg = float(gt_row['yaw_deg']) if 'yaw_deg' in imu_log.columns else 0.0

    hp_raw  = r.get('homo_position')                       # uncorrected
    hp_corr = _correct_lookahead(hp_raw, heading_deg, LOOKAHEAD_M)

    vq = r.get('visual_quality', {})
    bank_deg = math.degrees(abs(float(raw_row.get('bank', 0.0))))

    est_lat, est_lon = trajectory_est[i]
    bat_lat, bat_lon = trajectory_batch[i]

    frame_data.append({
        'frame':        i,
        'csv_idx':      csv_idx,
        'ts':           ts,
        'gt_lat':       gt_lat,
        'gt_lon':       gt_lon,
        'online_lat':   est_lat,
        'online_lon':   est_lon,
        'batch_lat':    bat_lat,
        'batch_lon':    bat_lon,
        'online_err':   ekf_errors_online[i],
        'batch_err':    ekf_errors_batch[i],
        'homo_err_raw': _hav(hp_raw[0],  hp_raw[1],  gt_lat, gt_lon) if hp_raw  else np.nan,
        'homo_err_corr':_hav(hp_corr[0], hp_corr[1], gt_lat, gt_lon) if hp_corr else np.nan,
        'gate_pass':    bool(r.get('gate_pass', False)),
        'CShape':       float(vq.get('CShape', 0)),
        'inliers':      int(vq.get('inliers', 0)),
        'sem_conf':     float(r.get('semantic_confidence') or 0.5),
        'method':       r.get('method', 'unknown'),
        'tiles_tested': int(r.get('tiles_tested', 0)),
        'search_time':  float(r.get('search_time', 0)),
        'verified':     bool(r.get('meta_tile_verified', False)),
        'bank_deg':     bank_deg,
        'in_map':       (map_bounds['lat_min'] <= gt_lat <= map_bounds['lat_max'] and
                         map_bounds['lon_min'] <= gt_lon <= map_bounds['lon_max']),
    })

df_results = pd.DataFrame(frame_data)
N = len(df_results)
print(f"{'='*78}")
print(f"  PIPELINE 3 — COMPREHENSIVE RESULTS  ({N} frames,  gate={gate_count}/{N})")
print(f"{'='*78}")

# ── Per-method statistics ──
method_cols = {
    'Online EKF  (visual+IMU)': 'online_err',
    'Batch EKF   (dead-reckon)': 'batch_err',
    'Homo raw    (uncorrected)': 'homo_err_raw',
    'Homo corr   (look-ahead) ': 'homo_err_corr',
}
thesh = [10, 25, 50, 100, 200, 500]
print(f"\n  {'Method':<28s} | {'n':>4s} {'mean':>7s} {'med':>7s} {'std':>7s} {'max':>7s} "
      f"| {'<10m':>5s} {'<25m':>5s} {'<50m':>5s} {'<100m':>5s} {'<200m':>5s}")
print(f"  {'-'*100}")
for lbl, col in method_cols.items():
    v = df_results[col].dropna().values
    if len(v) == 0: continue
    pcts = "  ".join(f"{(v<t).mean()*100:4.0f}%" for t in thesh[:5])
    print(f"  {lbl} | {len(v):4d} {np.mean(v):7.1f} {np.median(v):7.1f} "
          f"{np.std(v):7.1f} {np.max(v):7.0f} | {pcts}")

# Improvement
imp = df_results['batch_err'].mean() - df_results['online_err'].mean()
imp_pct = imp / df_results['batch_err'].mean() * 100
print(f"\n  Improvement vs batch:   {imp:+.1f}m  ({imp_pct:.0f}%)")
beats = (df_results['online_err'] < df_results['batch_err']).sum()
print(f"  Online beats batch on:  {beats}/{N} frames ({beats/N*100:.0f}%)")

# ── Percentile table ──
print(f"\n  {'Percentile':<28s} | {'P50':>8s} {'P75':>8s} {'P90':>8s} {'P95':>8s} {'P99':>8s}")
print(f"  {'-'*72}")
for lbl, col in method_cols.items():
    v = df_results[col].dropna().values
    if len(v) == 0: continue
    ps = [np.percentile(v, p) for p in [50, 75, 90, 95, 99]]
    print(f"  {lbl} | {ps[0]:8.1f} {ps[1]:8.1f} {ps[2]:8.1f} {ps[3]:8.1f} {ps[4]:8.1f}")

# ── Method distribution ──
print(f"\n  Method distribution: {dict(Counter(df_results['method']))}")
print(f"  Avg tiles tested: {df_results['tiles_tested'].mean():.1f}   "
      f"avg search time: {df_results['search_time'].mean():.2f}s/frame")
print(f"  Semantic verified: {df_results['verified'].sum()}/{N}")
print(f"  Turn frames (bank>10°): {(df_results['bank_deg']>10).sum()}/{N}")
print(f"  EKF P_pos final: [{ekf.P[8,8]:.0f}, {ekf.P[9,9]:.0f}] m²")

# ── In-Map vs Out-of-Map breakdown ──
in_map = df_results[df_results['in_map']]
out_map = df_results[~df_results['in_map']]
print(f"\n  {'─'*60}")
print(f"  IN-MAP REGION ANALYSIS ({len(in_map)}/{N} frames)")
print(f"  {'─'*60}")
if len(in_map) > 0:
    print(f"    Online EKF:  mean={in_map['online_err'].mean():.1f}m  "
          f"median={in_map['online_err'].median():.1f}m  "
          f"max={in_map['online_err'].max():.0f}m")
    print(f"    Batch  EKF:  mean={in_map['batch_err'].mean():.1f}m")
    imp_im = in_map['batch_err'].mean() - in_map['online_err'].mean()
    print(f"    Improvement: {imp_im:+.1f}m ({imp_im / in_map['batch_err'].mean() * 100:.0f}%)")
    print(f"    Gate pass:   {in_map['gate_pass'].sum()}/{len(in_map)} "
          f"({in_map['gate_pass'].mean()*100:.0f}%)")
    print(f"    <50m: {(in_map['online_err']<50).sum()}/{len(in_map)} "
          f"({(in_map['online_err']<50).mean()*100:.0f}%)  "
          f"<100m: {(in_map['online_err']<100).sum()}/{len(in_map)} "
          f"({(in_map['online_err']<100).mean()*100:.0f}%)")
if len(out_map) > 0:
    print(f"\n  OUT-OF-MAP ({len(out_map)} frames):")
    print(f"    Online EKF:  mean={out_map['online_err'].mean():.1f}m  "
          f"(no visual correction possible — EKF coasts on IMU)")


In [ ]:
# Cell 10 — Error CDF + Percentile Comparison Plot
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: CDF curves ──
ax = axes[0]
color_map = {
    'Online EKF':  ('blue',   'solid',  2.5,  df_results['online_err'].dropna().values),
    'Batch EKF':   ('red',    'dashed', 1.8,  df_results['batch_err'].dropna().values),
    'Homo raw':    ('green',  'dotted', 1.5,  df_results['homo_err_raw'].dropna().values),
    'Homo corr':   ('purple', 'dashdot',1.5,  df_results['homo_err_corr'].dropna().values),
}
for label, (color, ls, lw, vals) in color_map.items():
    if len(vals) == 0: continue
    xs = np.sort(vals)
    ys = np.linspace(0, 1, len(xs))
    ax.plot(xs, ys * 100, color=color, ls=ls, lw=lw, label=f"{label} (n={len(vals)})")

for thresh, col in [(10,'gray'), (25,'silver'), (50,'gold'), (100,'orange'), (200,'tomato')]:
    ax.axvline(thresh, color=col, ls=':', alpha=0.6, lw=1)
    ax.text(thresh + 1, 2, f'{thresh}m', color=col, fontsize=8, va='bottom')

ax.set_xlabel('Error (m)', fontsize=12)
ax.set_ylabel('Cumulative % of frames', fontsize=12)
ax.set_title('Error CDF — All Methods', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 500)
ax.set_ylim(0, 100)

# ── Right: Error over time (all frames) ──
ax2 = axes[1]
frames = df_results['frame'].values
ax2.plot(frames, df_results['online_err'], 'b-',  lw=1.5, label='Online EKF')
ax2.plot(frames, df_results['batch_err'],  'r--', lw=1.1, alpha=0.6, label='Batch EKF')
hp_frames = df_results[df_results['homo_err_corr'].notna()]['frame'].values
hp_errs   = df_results[df_results['homo_err_corr'].notna()]['homo_err_corr'].values
ax2.scatter(hp_frames, hp_errs, c='purple', s=15, alpha=0.5, zorder=5, label='Homo corr')
gated_frames = df_results[df_results['gate_pass']]['frame'].values
gated_errs   = df_results[df_results['gate_pass']]['online_err'].values
ax2.scatter(gated_frames, gated_errs, c='blue', s=30, alpha=0.35,
            zorder=4, marker='o', edgecolors='navy', linewidths=0.5, label='Gate pass (online)')
ax2.axhline(50,  color='gold',   ls='--', lw=1.5, alpha=0.7, label='50m target')
ax2.axhline(100, color='orange', ls='--', lw=1.0, alpha=0.5, label='100m')

# Shade bank-angle turns
turn_mask = df_results['bank_deg'] > 10
extra_handles = []
if turn_mask.any():
    in_turn = False
    for j, is_turn in enumerate(turn_mask):
        if is_turn and not in_turn:
            turn_start = j; in_turn = True
        elif not is_turn and in_turn:
            ax2.axvspan(turn_start, j, color='orange', alpha=0.12)
            in_turn = False
    if in_turn:
        ax2.axvspan(turn_start, len(turn_mask)-1, color='orange', alpha=0.12)
    extra_handles.append(Patch(facecolor='orange', alpha=0.12, label='Turn (bank>10°)'))

ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('Error (m)', fontsize=12)
ax2.set_title('Localization Error Over Time (all frames)', fontsize=14)
handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles=handles + extra_handles, fontsize=9, loc='upper left')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/error_cdf_and_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/error_cdf_and_timeline.png")

In [ ]:

# Cell 11 — Flight Phase Analysis: Straight vs Turn vs Post-turn
import matplotlib.pyplot as plt
import numpy as np

# Classify each frame by bank angle
# Straight: bank < 5°  |  Turn: bank 5-20°  |  Bank turn: bank > 20°
df = df_results.copy()
df['phase'] = pd.cut(df['bank_deg'], bins=[-1, 5, 20, 90],
                     labels=['straight', 'turn', 'heavy_turn'])

print(f"{'Phase':<14s} | {'N':>4s} | {'Online mean':>11s} | {'Batch mean':>10s} | "
      f"{'Homo corr mean':>14s} | {'Gate%':>6s} | {'<50m':>5s}")
print('-' * 78)
for ph in ['straight', 'turn', 'heavy_turn']:
    sub = df[df['phase'] == ph]
    if sub.empty: continue
    om  = sub['online_err'].mean()
    bm  = sub['batch_err'].mean()
    hm  = sub['homo_err_corr'].dropna().mean()
    gp  = sub['gate_pass'].mean() * 100
    lt50 = (sub['online_err'] < 50).mean() * 100
    print(f"  {ph:<12s} | {len(sub):4d} | {om:>10.1f}m | {bm:>9.1f}m | "
          f"{'N/A' if np.isnan(hm) else f'{hm:>13.1f}m'} | {gp:>5.0f}% | {lt50:>4.0f}%")

# ── Plot error by flight phase ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Error by phase (box plot)
ax = axes[0, 0]
phase_groups = [df[df['phase'] == ph]['online_err'].values
                for ph in ['straight', 'turn', 'heavy_turn']]
labels = ['Straight\n(bank<5°)', 'Turn\n(5-20°)', 'Heavy turn\n(>20°)']
bps = ax.boxplot([g for g, l in zip(phase_groups, labels) if len(g) > 0],
                 labels=[l for g, l in zip(phase_groups, labels) if len(g) > 0],
                 patch_artist=True, notch=False,
                 boxprops=dict(facecolor='lightblue', color='navy'),
                 medianprops=dict(color='red', lw=2))
ax.axhline(50,  color='gold',   ls='--', alpha=0.7, label='50m target')
ax.axhline(100, color='orange', ls='--', alpha=0.5)
ax.set_ylabel('Online EKF Error (m)'); ax.set_title('Error by Flight Phase')
ax.grid(True, alpha=0.3, axis='y'); ax.legend()

# Bank angle and error over time
ax2 = axes[0, 1]
ax2b = ax2.twinx()
ax2.plot(df['frame'], df['online_err'], 'b-', lw=1.5, label='Online err', alpha=0.8)
ax2b.fill_between(df['frame'], df['bank_deg'], alpha=0.25, color='orange', label='Bank angle')
ax2b.set_ylabel('Bank angle (°)', color='orange')
ax2b.tick_params(axis='y', labelcolor='orange')
ax2.set_xlabel('Frame'); ax2.set_ylabel('Error (m)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')
ax2.set_title('Error vs Bank Angle Over Time')
ax2.axhline(50, color='gold', ls='--', alpha=0.5)
ax2.grid(True, alpha=0.2)

# CShape over time  
ax3 = axes[1, 0]
ax3.plot(df['frame'], df['CShape'], 'g-o', ms=2.5, lw=1, label='CShape')
ax3.plot(df['frame'], df['inliers'] / df['inliers'].max().clip(1), 'm--', lw=1,
         alpha=0.6, label='Inliers (norm)')
ax3.fill_between(df['frame'], df['bank_deg'] / df['bank_deg'].max().clip(1),
                 alpha=0.15, color='orange', label='Bank (norm)')
ax3.axhline(0.3, color='red', ls='--', alpha=0.5, label='Gate thresh')
ax3.set_xlabel('Frame'); ax3.set_ylabel('Quality metric')
ax3.set_title('Visual Quality + Bank Over Time')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)
ax3.set_ylim(-0.05, 1.05)

# Semantic confidence over time
ax4 = axes[1, 1]
sem_vals = df['sem_conf'].fillna(0.5)
ax4.plot(df['frame'], df['online_err'], 'b-', lw=1.2, alpha=0.7, label='Online err')
sc = ax4.scatter(df['frame'], df['online_err'], c=sem_vals, cmap='RdYlGn',
                 s=20, vmin=0, vmax=1, zorder=5, label='Sem conf (colour)')
plt.colorbar(sc, ax=ax4, label='Semantic confidence', shrink=0.8)
ax4.axhline(50, color='gold', ls='--', alpha=0.5)
ax4.set_xlabel('Frame'); ax4.set_ylabel('Error (m)')
ax4.set_title('Error coloured by Semantic Confidence')
ax4.grid(True, alpha=0.3); ax4.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('../outputs/flight_phase_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/flight_phase_analysis.png")


In [ ]:

# Cell 12 — Optimal Look-ahead Calibration (data-driven LOOKAHEAD_M)
import math
import numpy as np
import matplotlib.pyplot as plt

# For each gate-passing frame, measure the raw (uncorrected) offset from homo→GT
# projected onto the heading vector.
raw_forward_offsets = []
for i, (csv_idx, ts, fp) in enumerate(aligned[:len(results)]):
    r = results[i]
    hp_raw = r.get('homo_position')
    if hp_raw is None or not r.get('gate_pass', False):
        continue
    gt_row = imu_log.iloc[csv_idx]
    gt_lat, gt_lon = float(gt_row['gps_lat']), float(gt_row['gps_lon'])
    heading_deg = float(gt_row['yaw_deg']) if 'yaw_deg' in imu_log.columns else 0.0

    # Offset vector in metres (homo_raw → GT)
    d_north = (gt_lat - hp_raw[0]) * 111320.0
    d_east  = (gt_lon - hp_raw[1]) * 111320.0 * math.cos(math.radians(gt_lat))

    # Project onto heading direction
    h = math.radians(heading_deg)
    fwd_n, fwd_e = math.cos(h), math.sin(h)
    forward_component = d_north * fwd_n + d_east * fwd_e  # positive = GT is ahead of homo → need positive lookahead
    lateral_component = -d_north * fwd_e + d_east * fwd_n  # positive = GT is right of homo

    raw_forward_offsets.append({
        'frame': i, 'forward': forward_component, 'lateral': lateral_component,
        'dist': math.sqrt(d_north**2 + d_east**2),
        'cs': r.get('visual_quality', {}).get('CShape', 0),
    })

if raw_forward_offsets:
    fwds = np.array([o['forward'] for o in raw_forward_offsets])
    lats = np.array([o['lateral'] for o in raw_forward_offsets])
    print(f"Gate-passing frames with homo position: {len(fwds)}")
    print(f"\nForward component (homo → GT, along heading):")
    print(f"  Mean:   {np.mean(fwds):+8.1f}m  (positive = GT is AHEAD → apply positive lookahead)")
    print(f"  Median: {np.median(fwds):+8.1f}m")
    print(f"  Std:    {np.std(fwds):8.1f}m")
    print(f"\nLateral component (homo → GT, perpendicular to heading):")
    print(f"  Mean:   {np.mean(lats):+8.1f}m  (should be ~0 if no lateral camera tilt)")
    print(f"  Std:    {np.std(lats):8.1f}m")
    opt_lookahead = np.median(fwds)
    print(f"\n{'='*60}")
    print(f"  OPTIMAL LOOKAHEAD_M = {opt_lookahead:.1f}m  (current: {LOOKAHEAD_M}m)")
    print(f"  Applying this would reduce mean forward bias by "
          f"{abs(np.mean(fwds) - opt_lookahead):.1f}m")
    print(f"{'='*60}")

    # Simulate different lookahead values
    test_deltas = np.arange(0, 250, 5)
    residual_mean = []
    residual_median = []
    for delta in test_deltas:
        # Corrected homo = raw_homo + lookahead backward along heading
        # forward_component is (GT - raw_homo) projected forward
        # after correction, new (GT - corrected) forward = forward_component - delta
        residuals = np.sqrt((fwds - delta)**2 + lats**2)
        residual_mean.append(np.mean(residuals))
        residual_median.append(np.median(residuals))

    best_mean_idx   = np.argmin(residual_mean)
    best_median_idx = np.argmin(residual_median)
    print(f"\n  Lookahead scan → best mean error:   {test_deltas[best_mean_idx]:.0f}m  "
          f"(residual={residual_mean[best_mean_idx]:.1f}m)")
    print(f"  Lookahead scan → best median error: {test_deltas[best_median_idx]:.0f}m  "
          f"(residual={residual_median[best_median_idx]:.1f}m)")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    ax.plot(test_deltas, residual_mean,   'b-',  lw=2, label='Mean residual')
    ax.plot(test_deltas, residual_median, 'r--', lw=2, label='Median residual')
    ax.axvline(LOOKAHEAD_M, color='gray', ls=':', lw=2, label=f'Current ({LOOKAHEAD_M}m)')
    ax.axvline(test_deltas[best_mean_idx], color='blue', ls='--', alpha=0.5,
               label=f'Optimal mean ({test_deltas[best_mean_idx]:.0f}m)')
    ax.set_xlabel('Lookahead correction (m)'); ax.set_ylabel('Mean/median residual error (m)')
    ax.set_title('Lookahead Calibration Scan'); ax.legend(); ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.scatter(fwds, lats, c=df_results[df_results['gate_pass']]['CShape'],
                cmap='RdYlGn', s=30, vmin=0, vmax=1, alpha=0.7)
    ax2.axvline(0, color='gray', ls='-', alpha=0.3)
    ax2.axhline(0, color='gray', ls='-', alpha=0.3)
    ax2.axvline(np.mean(fwds), color='blue', ls='--', lw=2, label=f'Mean fwd={np.mean(fwds):.0f}m')
    circle = plt.Circle((0, 0), 50, color='gold', fill=False, lw=2, ls='--', label='50m radius')
    ax2.add_patch(circle)
    ax2.set_xlabel('Forward offset (m)  [+ve = homo behind drone]')
    ax2.set_ylabel('Lateral offset (m)')
    ax2.set_title('Homo→GT offset in body frame (coloured by CShape)')
    ax2.legend(); ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    plt.colorbar(ax2.collections[0], ax=ax2, label='CShape')

    plt.tight_layout()
    plt.savefig('../outputs/lookahead_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/lookahead_calibration.png")
else:
    print("No gate-passing frames available for look-ahead calibration.")


In [ ]:

# Cell 13 — Geographic Error Heatmap + Trajectory Quality Map
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# ── Left: Error magnitude heat map on trajectory ──
ax = axes[0]
# GT trajectory
ax.plot([p[1] for p in trajectory_gt], [p[0] for p in trajectory_gt],
        'g-', lw=0.8, alpha=0.4, label='GT path', zorder=1)

# Scatter coloured by online error (capped at 300m for visibility)
errs = np.array(ekf_errors_online)
errs_clipped = np.clip(errs, 0, 300)
sc = ax.scatter([p[1] for p in trajectory_est], [p[0] for p in trajectory_est],
                c=errs_clipped, cmap='RdYlGn_r', s=18, vmin=0, vmax=300,
                zorder=3, alpha=0.85, label='Online EKF (colour=error)')
plt.colorbar(sc, ax=ax, label='Online EKF error (m, capped 300m)', shrink=0.8)

ax.scatter([trajectory_gt[0][1]], [trajectory_gt[0][0]],
           c='green', s=150, marker='^', zorder=5, edgecolors='k', label='Start')
ax.scatter([trajectory_gt[-1][1]], [trajectory_gt[-1][0]],
           c='red', s=150, marker='v', zorder=5, edgecolors='k', label='End')

# Reference map tile boundary
ax.axhline(map_bounds['lat_min'], color='gray', ls=':', alpha=0.3)
ax.axhline(map_bounds['lat_max'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_min'], color='gray', ls=':', alpha=0.3)
ax.axvline(map_bounds['lon_max'], color='gray', ls=':', alpha=0.3)
ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('Online EKF Error Heatmap on Trajectory', fontsize=14)
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# ── Right: Gate pass map + homo position cloud ──
ax2 = axes[1]
# All frames — colour by whether gate passed
gated = df_results[df_results['gate_pass']]
ungated = df_results[~df_results['gate_pass']]

ax2.scatter(ungated['online_lon'], ungated['online_lat'],
            c='gray', s=12, alpha=0.4, label='Gate fail (PF/EKF coast)')
ax2.scatter(gated['online_lon'], gated['online_lat'],
            c=gated['online_err'].clip(0, 200), cmap='RdYlGn_r',
            s=25, alpha=0.8, vmin=0, vmax=200, zorder=3,
            label='Gate pass (visual update, colour=err)')
ax2.plot([p[1] for p in trajectory_gt], [p[0] for p in trajectory_gt],
         'g-', lw=1.5, alpha=0.6, label='GT path', zorder=2)

# Homo position estimates (raw)
hp_lats = [r['homo_position'][0] for r in results if r.get('homo_position')]
hp_lons = [r['homo_position'][1] for r in results if r.get('homo_position')]
if hp_lats:
    ax2.scatter(hp_lons, hp_lats, c='blue', s=8, alpha=0.2,
                zorder=2, label=f'Homo raw ({len(hp_lats)} frames)', marker='+')

ax2.set_xlabel('Longitude (°E)', fontsize=12)
ax2.set_ylabel('Latitude (°N)', fontsize=12)
ax2.set_title('Gate Pass Map + Visual Measurement Cloud', fontsize=14)
ax2.legend(fontsize=9, loc='lower left')
ax2.grid(True, alpha=0.3); ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('../outputs/geographic_error_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/geographic_error_map.png")


In [ ]:

# Cell 14 — Semantic Confidence & Gate Analysis
import matplotlib.pyplot as plt
import numpy as np

df = df_results.copy()

# ── Semantic confidence bins vs accuracy ──
sc_bins = [0.0, 0.3, 0.5, 0.7, 0.9, 1.01]
sc_labels = ['<0.3', '0.3-0.5', '0.5-0.7', '0.7-0.9', '>0.9']
df['sc_bin'] = pd.cut(df['sem_conf'], bins=sc_bins, labels=sc_labels, right=False)

print("Semantic Confidence vs Accuracy:")
print(f"  {'SC bin':<10s} | {'N':>4s} {'Online err':>10s} | {'Batch err':>9s} | {'Gate%':>6s}")
print(f"  {'-'*55}")
for sc_lbl in sc_labels:
    sub = df[df['sc_bin'] == sc_lbl]
    if sub.empty: continue
    oe = sub['online_err'].mean()
    be = sub['batch_err'].mean()
    gp = sub['gate_pass'].mean() * 100
    print(f"  {sc_lbl:<10s} | {len(sub):4d}  {oe:9.1f}m | {be:8.1f}m | {gp:5.0f}%")

# ── Gate pass/fail breakdown ──
gated   = df[df['gate_pass']]
ungated = df[~df['gate_pass']]
print(f"\nGate PASS ({len(gated)} frames):")
print(f"  Online err: mean={gated['online_err'].mean():.1f}m  median={gated['online_err'].median():.1f}m")
print(f"  Homo corr err (at update): mean={gated['homo_err_corr'].dropna().mean():.1f}m")
print(f"\nGate FAIL ({len(ungated)} frames):")
print(f"  Online err: mean={ungated['online_err'].mean():.1f}m  median={ungated['online_err'].median():.1f}m")
if ungated['homo_err_corr'].notna().any():
    print(f"  Homo corr err (not used): mean={ungated['homo_err_corr'].dropna().mean():.1f}m")

# ── CShape vs accuracy scatter ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
has_homo = df['homo_err_corr'].notna()
sc = ax.scatter(df[has_homo]['CShape'], df[has_homo]['homo_err_corr'].clip(0, 500),
                c=df[has_homo]['gate_pass'].astype(int), cmap='RdYlGn',
                s=20, alpha=0.7, vmin=0, vmax=1)
ax.axvline(0.3, color='red', ls='--', alpha=0.6, label='Gate thresh')
ax.axvline(0.5, color='orange', ls='--', alpha=0.5, label='High-quality')
ax.axhline(50, color='gold', ls='--', alpha=0.5)
ax.set_xlabel('CShape'); ax.set_ylabel('Homo corr error (m, clip 500)')
ax.set_title('CShape vs Homo Error (green=gate pass)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Gate pass')

ax2 = axes[0, 1]
ax2.scatter(df['inliers'].clip(0, 700), df['online_err'].clip(0, 300),
            c=df['gate_pass'].astype(int), cmap='RdYlGn', s=15, alpha=0.5)
ax2.axvline(20,  color='red',    ls='--', alpha=0.6, label='Gate thresh')
ax2.axvline(100, color='orange', ls='--', alpha=0.5)
ax2.axhline(50, color='gold', ls='--', alpha=0.5)
ax2.set_xlabel('Inliers (clip 700)'); ax2.set_ylabel('Online EKF error (m, clip 300)')
ax2.set_title('Inliers vs Online Error')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3.scatter(df['sem_conf'], df['online_err'].clip(0, 300),
            c=df['gate_pass'].astype(int), cmap='RdYlGn', s=15, alpha=0.5)
ax3.set_xlabel('Semantic Confidence'); ax3.set_ylabel('Online EKF error (m, clip 300)')
ax3.set_title('Semantic Confidence vs Online Error')
ax3.grid(True, alpha=0.3); ax3.axhline(50, color='gold', ls='--', alpha=0.5)

ax4 = axes[1, 1]
# Histogram of semantic confidence
ax4.hist(df['sem_conf'].dropna(), bins=30, color='steelblue', alpha=0.7, label='All frames')
ax4.hist(df[df['gate_pass']]['sem_conf'].dropna(), bins=30, color='green',
         alpha=0.5, label='Gate pass')
ax4.set_xlabel('Semantic Confidence'); ax4.set_ylabel('Count')
ax4.set_title('Semantic Confidence Distribution')
ax4.legend(); ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/semantic_gate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/semantic_gate_analysis.png")


In [ ]:

# Cell 15 — Per-Frame Comparison Table (top 30 worst + best)
import numpy as np

print("TOP 25 WORST ONLINE EKF FRAMES:")
print(f"{'F':>4s} {'time':>6s} {'online':>8s} {'batch':>7s} {'homo_c':>7s} | "
      f"{'CS':>5s} {'inl':>5s} {'sem':>5s} | {'gate':>4s} {'bank':>5s}° | {'method':<18s}")
print('-' * 90)
worst = df_results.nlargest(25, 'online_err')
for _, r in worst.iterrows():
    hc = f"{r['homo_err_corr']:6.0f}m" if not np.isnan(r['homo_err_corr']) else "   N/A"
    print(f"F{r['frame']:3.0f} {r['ts']:6.0f}s | "
          f"{r['online_err']:6.0f}m {r['batch_err']:6.0f}m {hc} | "
          f"{r['CShape']:5.3f} {r['inliers']:5.0f} {r['sem_conf']:5.2f} | "
          f"{'Y' if r['gate_pass'] else 'n':>4s} {r['bank_deg']:5.1f}  | "
          f"{r['method']:<18s}")

print(f"\nTOP 25 BEST ONLINE EKF FRAMES:")
print(f"{'F':>4s} {'time':>6s} {'online':>8s} {'batch':>7s} {'homo_c':>7s} | "
      f"{'CS':>5s} {'inl':>5s} {'sem':>5s} | {'gate':>4s} {'bank':>5s}° | {'method':<18s}")
print('-' * 90)
best = df_results.nsmallest(25, 'online_err')
for _, r in best.iterrows():
    hc = f"{r['homo_err_corr']:6.0f}m" if not np.isnan(r['homo_err_corr']) else "   N/A"
    print(f"F{r['frame']:3.0f} {r['ts']:6.0f}s | "
          f"{r['online_err']:6.0f}m {r['batch_err']:6.0f}m {hc} | "
          f"{r['CShape']:5.3f} {r['inliers']:5.0f} {r['sem_conf']:5.2f} | "
          f"{'Y' if r['gate_pass'] else 'n':>4s} {r['bank_deg']:5.1f}  | "
          f"{r['method']:<18s}")

# ── Print where online is WORSE than batch ──
worse_mask = df_results['online_err'] > df_results['batch_err']
worse_df = df_results[worse_mask].sort_values('online_err', ascending=False)
print(f"\nFrames where ONLINE is WORSE than BATCH ({len(worse_df)}/{len(df_results)}):")
for _, r in worse_df.head(15).iterrows():
    diff = r['online_err'] - r['batch_err']
    print(f"  F{r['frame']:3.0f}: online={r['online_err']:5.0f}m  batch={r['batch_err']:5.0f}m  "
          f"delta={diff:+5.0f}m  CS={r['CShape']:.3f}  bank={r['bank_deg']:.1f}°  "
          f"gate={'Y' if r['gate_pass'] else 'n'}")


In [ ]:

# Cell 16 — Timing / Performance Breakdown
import numpy as np
import matplotlib.pyplot as plt

search_times = np.array(df_results['search_time'].values)
print(f"Search time per frame:")
print(f"  Mean:   {search_times.mean():.3f}s")
print(f"  Median: {np.median(search_times):.3f}s")
print(f"  Min:    {search_times.min():.3f}s")
print(f"  Max:    {search_times.max():.3f}s")
print(f"  P90:    {np.percentile(search_times, 90):.3f}s")
print(f"  Total:  {search_times.sum():.1f}s for {len(search_times)} frames")
print(f"  FPS:    {1.0 / search_times.mean():.2f}")

print(f"\nTile search efficiency:")
print(f"  Avg tiles tested:  {df_results['tiles_tested'].mean():.1f}")
print(f"  Max tiles tested:  {df_results['tiles_tested'].max()}")
print(f"  Frames with 0 tiles: {(df_results['tiles_tested'] == 0).sum()} (imu_fallback)")

by_method = df_results.groupby('method')['search_time'].agg(['mean', 'count'])
print(f"\nSearch time by method:")
for m, row in by_method.iterrows():
    print(f"  {m:<22s}: mean={row['mean']:.3f}s  n={int(row['count'])}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(df_results['frame'], df_results['search_time'], 'b-', lw=1.2, alpha=0.8)
ax.axhline(search_times.mean(), color='red', ls='--', label=f'Mean={search_times.mean():.2f}s')
ax.fill_between(df_results['frame'], 0, df_results['search_time'],
                alpha=0.15, color='blue')
ax.set_xlabel('Frame'); ax.set_ylabel('Time (s)'); ax.set_title('Search Time Per Frame')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

ax2 = axes[1]
ax2.hist(search_times, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(search_times.mean(), color='red', ls='--', lw=2,
            label=f'Mean={search_times.mean():.2f}s')
ax2.axvline(np.percentile(search_times, 90), color='orange', ls='--', lw=2,
            label=f'P90={np.percentile(search_times, 90):.2f}s')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Count'); ax2.set_title('Time Distribution')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')

ax3 = axes[2]
ax3.scatter(df_results['tiles_tested'], df_results['search_time'], 
            c=df_results['online_err'].clip(0, 200), cmap='RdYlGn_r',
            s=20, alpha=0.6, vmin=0, vmax=200)
ax3.set_xlabel('Tiles tested'); ax3.set_ylabel('Search time (s)')
ax3.set_title('Tiles Tested vs Search Time')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/timing_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/timing_analysis.png")


In [ ]:
# Cell 17 — SP+LG Feature Matching vs Semantic Model: Contribution Analysis
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# ── 1. CShape vs Semantic Confidence scatter, colored by online error ──
ax = axes[0, 0]
mask_valid = df_results['online_err'].notna()
sc = ax.scatter(df_results.loc[mask_valid, 'CShape'],
                df_results.loc[mask_valid, 'sem_conf'],
                c=df_results.loc[mask_valid, 'online_err'].clip(0, 150),
                cmap='RdYlGn_r', s=25, alpha=0.7, vmin=0, vmax=150)
ax.axvline(0.3, color='red', ls='--', lw=1, alpha=0.6, label='CShape gate (0.3)')
ax.axhline(0.5, color='blue', ls='--', lw=1, alpha=0.6, label='Sem conf 0.5')
ax.set_xlabel('CShape (SP+LG quality)', fontsize=11)
ax.set_ylabel('Semantic Confidence (histogram)', fontsize=11)
ax.set_title('SP+LG Quality vs Semantic Confidence', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Online EKF error (m)', shrink=0.8)

# ── 2. Agreement/disagreement analysis ──
ax = axes[0, 1]
high_sp = df_results['CShape'] > 0.3
high_sem = df_results['sem_conf'] > 0.6
quadrants = {
    'Both High\n(SP+LG + Sem agree)':   high_sp & high_sem,
    'SP+LG High only\n(Sem low)':       high_sp & ~high_sem,
    'Sem High only\n(SP+LG low)':       ~high_sp & high_sem,
    'Both Low\n(both disagree)':         ~high_sp & ~high_sem,
}
colors_q = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
labels_q = []
means_q = []
counts_q = []
for (lbl, mask), col in zip(quadrants.items(), colors_q):
    errs = df_results.loc[mask, 'online_err']
    labels_q.append(lbl)
    means_q.append(errs.mean() if len(errs) > 0 else 0)
    counts_q.append(len(errs))

bars = ax.bar(range(len(labels_q)), means_q, color=colors_q, alpha=0.8, edgecolor='white')
for i, (b, n) in enumerate(zip(bars, counts_q)):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
            f'n={n}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(labels_q)))
ax.set_xticklabels(labels_q, fontsize=8)
ax.set_ylabel('Mean Online EKF Error (m)', fontsize=11)
ax.set_title('Agreement Quadrant Analysis', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(50, color='gold', ls='--', lw=1.5, alpha=0.7, label='50m target')
ax.legend(fontsize=8)

# ── 3. SP+LG inliers vs error, colored by semantic confidence ──
ax = axes[0, 2]
gate_mask = df_results['gate_pass']
sc2 = ax.scatter(df_results.loc[gate_mask, 'inliers'],
                 df_results.loc[gate_mask, 'online_err'].clip(0, 200),
                 c=df_results.loc[gate_mask, 'sem_conf'],
                 cmap='RdYlGn', s=25, alpha=0.7, vmin=0.3, vmax=1.0)
ax.set_xlabel('SP+LG Inliers', fontsize=11)
ax.set_ylabel('Online EKF Error (m)', fontsize=11)
ax.set_title('Inliers vs Error (gate-pass only)', fontsize=12)
plt.colorbar(sc2, ax=ax, label='Semantic Confidence', shrink=0.8)
ax.axhline(50, color='gold', ls='--', lw=1.5, alpha=0.7)
ax.grid(True, alpha=0.3)

# ── 4. Time series: SP+LG CShape and Semantic Confidence (overlaid) ──
ax = axes[1, 0]
ax.plot(df_results['frame'], df_results['CShape'], 'b-', lw=1.2, alpha=0.8, label='CShape (SP+LG)')
ax.plot(df_results['frame'], df_results['sem_conf'], 'g-', lw=1.2, alpha=0.8, label='Sem Conf')
ax.axhline(0.3, color='red', ls=':', lw=1, alpha=0.5, label='CShape gate')
# Shade gate-fail frames
fail_mask = ~df_results['gate_pass']
for j, fail in enumerate(fail_mask):
    if fail:
        ax.axvspan(j-0.5, j+0.5, color='red', alpha=0.05)
ax.set_xlabel('Frame', fontsize=11)
ax.set_ylabel('Confidence Score', fontsize=11)
ax.set_title('SP+LG vs Semantic Confidence Over Time', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# ── 5. Measurement noise R modulation effect ──
ax = axes[1, 1]
R_base = np.where(
    (df_results['CShape'] > 0.5) & (df_results['inliers'] > 100),
    30.0**2, 60.0**2)
sem_factor = np.maximum(0.5, 2.0 - 1.5 * df_results['sem_conf'])
R_effective = R_base * sem_factor
turn_factor = np.where(df_results['bank_deg'] > np.degrees(TURN_ROLL_THRESHOLD_RAD),
                       TURN_R_MULTIPLIER, 1.0)
R_effective *= turn_factor
R_sqrt = np.sqrt(R_effective)
ax.plot(df_results['frame'], R_sqrt, 'purple', lw=1.2, alpha=0.8, label='√R effective')
ax.fill_between(df_results['frame'], 0, R_sqrt, alpha=0.15, color='purple')
ax2b = ax.twinx()
ax2b.plot(df_results['frame'], df_results['online_err'].clip(0, 200),
          'b-', lw=0.8, alpha=0.5, label='Online error')
ax2b.set_ylabel('Error (m)', fontsize=11, color='blue')
ax.set_xlabel('Frame', fontsize=11)
ax.set_ylabel('√R measurement noise (m)', fontsize=11, color='purple')
ax.set_title('Adaptive Measurement Noise Modulation', fontsize=12)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# ── 6. Summary: Which system drives accuracy? ──
ax = axes[1, 2]
# For gate-pass frames: correlate with error
gated_df = df_results[df_results['gate_pass']].copy()
if len(gated_df) > 10:
    from scipy import stats
    corr_sp, p_sp = stats.spearmanr(gated_df['CShape'], gated_df['online_err'])
    corr_sem, p_sem = stats.spearmanr(gated_df['sem_conf'], gated_df['online_err'])
    corr_inl, p_inl = stats.spearmanr(gated_df['inliers'], gated_df['online_err'])
    
    feats = ['CShape\n(SP+LG)', 'Inliers\n(SP+LG)', 'Sem Conf\n(Semantic)']
    corrs = [corr_sp, corr_inl, corr_sem]
    colors_c = ['#3498db', '#2980b9', '#27ae60']
    bars2 = ax.barh(range(len(feats)), corrs, color=colors_c, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats, fontsize=10)
    ax.set_xlabel('Spearman Correlation with Error\n(negative = higher quality → lower error)', fontsize=10)
    ax.set_title('Feature Importance: Who Drives Accuracy?', fontsize=12)
    for i, (b, c, p) in enumerate(zip(bars2, corrs, [p_sp, p_inl, p_sem])):
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.text(c + 0.01 if c > 0 else c - 0.01, i,
                f'ρ={c:.3f} {sig}', va='center',
                ha='left' if c > 0 else 'right', fontsize=9)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(-0.5, 0.3)
else:
    ax.text(0.5, 0.5, 'Insufficient gate-pass\nframes for analysis',
            ha='center', va='center', transform=ax.transAxes, fontsize=14)

plt.suptitle('SuperPoint+LightGlue vs Semantic Model — Contribution Analysis',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/splg_vs_semantic_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print("\n" + "="*78)
print("  SP+LG vs SEMANTIC MODEL — CONTRIBUTION SUMMARY")
print("="*78)
for lbl, mask in quadrants.items():
    n = mask.sum()
    if n == 0: continue
    errs = df_results.loc[mask, 'online_err']
    lt50 = (errs < 50).mean() * 100
    print(f"  {lbl.replace(chr(10), ' '):<35s}: n={n:3d}  mean={errs.mean():6.1f}m  "
          f"<50m={lt50:4.0f}%")

print(f"\n  SP+LG is the PRIMARY position estimator (homography → lat/lon).")
print(f"  Semantic model SUPPORTS by:")
print(f"    1. Pre-filtering tiles (reduces ~24 candidates → top-10 before SP+LG)")
print(f"    2. Modulating measurement noise R (high sem_conf → trust more)")
print(f"    3. Double-confirmation (logs terrain match for diagnostics)")
if len(gated_df) > 10:
    print(f"\n  Correlation with error (gate-pass frames):")
    print(f"    CShape (SP+LG):     ρ={corr_sp:+.3f}  {'(significant)' if p_sp < 0.05 else '(not significant)'}")
    print(f"    Inliers (SP+LG):    ρ={corr_inl:+.3f}  {'(significant)' if p_inl < 0.05 else '(not significant)'}")
    print(f"    Sem Conf (Semantic): ρ={corr_sem:+.3f}  {'(significant)' if p_sem < 0.05 else '(not significant)'}")
print("Saved: outputs/splg_vs_semantic_comparison.png")